In [0]:
# =============================================================================
# Notebook  : 02b_map_11_audiences_conversion
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_11_audiences_conversion
# Spec Ref  : §1.7 Audiences and Conversion (POC stubs)
# Purpose   : Initialise email_audience and email_model_conversion tables.
#             These are tenant-configurable — audience assignment rules
#             vary per client. Tables are created empty for POC.
#             Real audience logic added in a later phase.
#
# Run after : map_08, map_09, map_10 (all aggregations complete)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 11: audiences + conversion stubs ===  customer={customer_id}")

In [0]:
# CELL 3 -- email_audience: maps each email to audience segment(s)
# Tenant-configurable: inbound, outbound, product_qualified, etc.
create_map_table(
    f"{sv}.email_audience",
    """
        email          STRING NOT NULL,
        audience_name  STRING NOT NULL,
        tenant         BIGINT
    """,
    cluster_by="email"
)
print(f"  email_audience table ready (POC stub — populate with tenant rules)")

In [0]:
# CELL 4 -- email_model_conversion: tracks conversion events per email
# Supports multiple conversion models: Closed Won, SQL, MQL, etc.
create_map_table(
    f"{sv}.email_model_conversion",
    """
        email             STRING NOT NULL,
        conversion_model  STRING NOT NULL,
        conversion_date   DATE,
        amount            DOUBLE,
        tenant            BIGINT
    """,
    cluster_by="email, conversion_model"
)
print(f"  email_model_conversion table ready (POC stub)")

In [0]:
# CELL 5 -- Final summary of all map layer output tables
print("\n=== Map Layer Complete — All Output Tables ===")
map_tables = [
    ("contacts_all",            "Materialized: contacts unified view"),
    ("accounts_all",            "Materialized: accounts unified view"),
    ("crm_events",              "SF Tasks → standardized meta_events"),
    ("mapped_events",           "All events with resolved contactid"),
    ("contacts_to_accounts",    "3-phase contact-account relationships"),
    ("accounts_attributes",     "Account is_paying/is_excluded/mrr"),
    ("contacts_attributes",     "Contact is_paying/is_excluded/mrr"),
    ("email_events_mapped",     "Email x meta_event x day aggregation"),
    ("domain_events_mapped",    "Domain x meta_event x day aggregation"),
    ("mk_account_events_mapped","Account rollup: identified + anonymous"),
    ("email_audience",          "Audience segment membership (stub)"),
    ("email_model_conversion",  "Conversion tracking (stub)"),
]
for tbl, desc in map_tables:
    try:
        n = cnt(f"{sv}.{tbl}")
        print(f"  {tbl:<35} {n:>10,} rows  |  {desc}")
    except Exception as e:
        print(f"  {tbl:<35}      ERROR  |  {e}")

dbutils.notebook.exit("Success")